# Heritage Short: Wan 2.2 TI2V-5B moving shots

Turns each exported keyframe into a ~5 s clip (image-to-video), using the motion prompt from `storyboard.json`.

**Before running**
1. Locally: `py -m pipeline shots --export outputs/kaggle/shots_job`, then upload that folder as a **Kaggle dataset** and attach it here.
2. Settings: **GPU** accelerator (T4 x2 or P100), **Internet on** (phone-verified account).

**Outputs:** `/kaggle/working/out/<shot>_<hash>.mp4` (the names come from the pipeline's cache contract; don't rename them) and `runs_fragment.jsonl`.
Download `out.zip`, unzip it, then run locally: `py -m pipeline shots --import <folder>`.
Clips whose names don't match the current storyboard are ignored on import; those shots keep their Ken Burns move.

**Budget:** a T4 is much slower than a 4090, so start with `MAX_SHOTS = 1`, read the time, then raise it.
Every shot logs its GPU seconds to `runs_fragment.jsonl` (used for COSTS.md).

In [ ]:
import json, time, datetime, subprocess, shutil, glob
from pathlib import Path

RUN_ID = "kaggle-" + datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
FRAGMENT = OUT / "runs_fragment.jsonl"

def log(stage, shot, model, params, seed, status, latency, output=None, error=None):
    """One line in the pipeline's runs.jsonl format; `--import` merges it locally."""
    rec = {"ts": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"), "run_id": RUN_ID,
           "stage": stage, "shot": shot, "provider": "kaggle", "model": model, "params": params,
           "seed": seed, "input_hash": None, "status": status, "attempt": 1,
           "latency_s": round(latency, 3), "gpu_s": round(latency, 3), "cost_usd_list": None,
           "output": output, "error": (error or None) and str(error)[:500]}
    with open(FRAGMENT, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def find_job():
    hits = glob.glob("/kaggle/input/**/job.json", recursive=True)
    assert hits, "attach the exported job folder as a Kaggle dataset (it must contain job.json)"
    path = Path(hits[0]); return path.parent, json.loads(path.read_text())

JOB_DIR, JOB = find_job()
MAX_SHOTS = 1          # raise after the first timing
SIZE = JOB["params"]["size"]            # "1280*704" (TI2V-5B 720p landscape)
FRAME_NUM = JOB["params"]["frame_num"]  # 121 frames = 5.04 s at 24 fps
print(len(JOB["jobs"]), "shot(s) in the job:", [j["shot"] for j in JOB["jobs"]])

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/Wan-Video/Wan2.2.git /kaggle/working/Wan2.2
# flash_attn needs a newer GPU than a T4 / P100: skip it, Wan falls back to standard attention
!grep -v flash_attn /kaggle/working/Wan2.2/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
!pip install -q "huggingface_hub[cli]"
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir /kaggle/working/Wan2.2-TI2V-5B

**Check the interface before spending GPU time.** The next cell prints `generate.py --help`.
The generation cell uses `--task --size --ckpt_dir --image --prompt --base_seed --frame_num --save_file --offload_model --convert_model_dtype --t5_cpu` (from the Wan 2.2 README).
If a flag was renamed, fix it in `wan_cmd()` below and record the change in FAILURES.md.

In [ ]:
!cd /kaggle/working/Wan2.2 && python generate.py --help | head -60

In [ ]:
def wan_cmd(job):
    return ["python", "generate.py", "--task", "ti2v-5B", "--size", SIZE,
            "--ckpt_dir", "/kaggle/working/Wan2.2-TI2V-5B",
            "--offload_model", "True", "--convert_model_dtype", "--t5_cpu",   # fit a 16 GB GPU
            "--image", str(JOB_DIR / job["image"]), "--prompt", job["prompt"],
            "--base_seed", str(job["seed"]), "--frame_num", str(FRAME_NUM),
            "--save_file", str(OUT / job["out"])]

for job in JOB["jobs"][:MAX_SHOTS]:
    start = time.time()
    r = subprocess.run(wan_cmd(job), cwd="/kaggle/working/Wan2.2", capture_output=True, text=True)
    took = time.time() - start
    ok = r.returncode == 0 and (OUT / job["out"]).exists()
    log("shots", job["shot"], JOB["model"], JOB["params"], job["seed"], "ok" if ok else "error",
        took, job["out"] if ok else None, None if ok else r.stderr[-800:])
    print(job["shot"], "ok" if ok else "FAILED", f"{took/60:.1f} min")
    if not ok:
        print(r.stderr[-1500:])

In [ ]:
shutil.make_archive("/kaggle/working/out", "zip", OUT)
print(sorted(p.name for p in OUT.iterdir()))
print("Download /kaggle/working/out.zip (Output tab), unzip, then: py -m pipeline shots --import <folder>")